# Hugging Face Applications — Lesson 1: Text Generation

> Learning material for **Hugging Face Applications**. Companion to the lesson script `01_Text_Generation.py` (same content, runnable without Jupyter).

**Task ID:** HF-201  |  **Folder:** `documentation`


## What is text generation?

A **text generation model** continues a piece of text: you give it a starting sentence, and it writes the next word, then the next, and so on.

Example — prompt `"Once upon a time"` might become:

> *Once upon a time* **there lived a small cat who dreamed of flying.**

## The core idea: predict the next word

Causal language models (like GPT-2) do exactly one thing:

1. Read the words so far.
2. Compute a **probability for every possible next word**.
3. Pick one (greedy: the most likely; or sample: randomly, weighted).
4. Append it, and repeat.

> **Analogy:** a person finishing a sentence in a conversation. The more
> words they hear, the better they guess what comes next.

## The pipeline shortcut

Hugging Face wraps all of that in one object: `pipeline("text-generation")`.

**Step 1 — load the model** (first run downloads ~350 MB, then cached):


In [ ]:
from transformers import pipeline

# distilgpt2 is the small, fast cousin of GPT-2 — perfect for learning on CPU.
generator = pipeline("text-generation", model="distilgpt2", device=-1)


`device=-1` means *run on CPU*. Remove it (or use 0) on machines with a GPU.

**Step 2 — generate!** The pipeline returns a list of dictionaries;
each has one key, `generated_text`, with the full prompt + continuation:


In [ ]:
result = generator("Once upon a time", max_new_tokens=30)
print(result[0]["generated_text"])


## Greedy vs. sampling

There are two ways to choose the next word:

| Decoding | Behavior | Output | 
|----------|----------|--------|
| **greedy** | always pick the most probable word | deterministic, often boring, can loop | 
| **sampling** | draw randomly from the probabilities | varied, creative, sometimes nonsense |

`do_sample=True` turns sampling on. Let's generate 3 different continuations
of the same prompt:


In [ ]:
outputs = generator(
    "Artificial intelligence will change the world because",
    max_new_tokens=40,
    num_return_sequences=3,   # ask for 3 continuations at once
    do_sample=True,
    temperature=1.0,
    top_k=50,
    top_p=0.95,
)

for i, out in enumerate(outputs, 1):
    print(f"--- continuation {i} ---")
    print(out["generated_text"])
    print()


## Temperature: the creativity dial

**Temperature** reshapes the probabilities before sampling:

- `temperature=0.2` → probabilities become extreme → output is focused, predictable, repetitive.
- `temperature=0.7` → a good balance (common default).
- `temperature=1.5` → probabilities flatten → output is wild and often nonsensical.

Run the same prompt at three temperatures and compare:


In [ ]:
prompt = "The best thing about Saturday morning is"

for temp in (0.2, 0.7, 1.5):
    out = generator(prompt, max_new_tokens=20, do_sample=True, temperature=temp, top_k=50)[0]
    print(f"temperature={temp}:")
    print(f"  {out['generated_text']}")
    print()


## Try it yourself

1. Generate a story with your own prompt (`"Last night I dreamt that"` ...).
2. Make the model write longer (`max_new_tokens=100`) — what starts to break?
3. Compare `do_sample=True` vs. default (greedy) on the same prompt.
4. **Bonus:** switch to `model="gpt2"` (bigger, better, slower).

## Common pitfalls

- **Download is slow on first run** — normal; it is cached afterwards.
- **Text degrades over length** — models repeat themselves; that's why `max_new_tokens` matters.
- **"CUDA out of memory"** — use `device=-1` (CPU) or a smaller model.

## Summary

- Text generation = predicting the next word, one at a time.
- `pipeline("text-generation")` hides tokenizer + model + decoding.
- Greedy is deterministic; sampling is creative; temperature controls how creative.

**Next lesson:** HF-202 — Summarization.  |  Extra reading: `../resources/reference_links.md`
